# TER Picopatt - Localisation

Importation des librairies principales et définition des dossiers.

Nos fonctions utilisées pour lire les données sont dans le fichier `functions.py`

In [42]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import radians, sin, cos, sqrt, atan2
import json
import geopandas as gpd
from shapely.geometry import shape, LineString, mapping
from ipywidgets import Output, VBox, HTML
from IPython.display import display
from ipyleaflet import Map, GeoJSON, GeomanDrawControl, LayersControl, basemaps, basemap_to_tiles

import functions as fc

# Dossiers de données et de sortie
DATA_NOZERO = Path("outputs/clean_nozeros")
FIG_FONT = Path("outputs/figures/fontaines")
OUT_LOCALISATION = Path("outputs/localisation")
fc.create_folder(OUT_LOCALISATION)
fc.create_folder(FIG_FONT)

# Affichage complet des colonnes
pd.set_option("display.max_columns", 200)

Lecture de tous les fichiers de données nettoyées (.csv, .xlsx) du dossier `clean_nozeros` dans un seul tableau, nettoyage des colonnes, puis vérification de la couverture temporelle et la répartition des mesures et ajout des colonnes `M_slot` (créneau horaire) et `date`.

In [43]:
bd = fc.load_all(DATA_NOZERO, False)

# Résumé
print("Chargement terminé")
print("Couverture :", bd['date'].min(), "->", bd['date'].max())
print("Parcours :", bd['track_id'].dropna().unique())
print("\nNombre de mesures par M_slot et parcours :")
print(
    bd.pivot_table(index="track_id", columns="M_slot", values="fichier_originaire", aggfunc="count")
       .fillna(0)
       .astype(int)
)

Chargement terminé
Couverture : 2024-10-29 -> 2025-01-16
Parcours : ['antigone' 'boulevards' 'ecusson']

Nombre de mesures par M_slot et parcours :
M_slot         M1     M2     M3     M4
track_id                              
antigone    29468  29924  31828  30237
boulevards  25603  34594  25785  29980
ecusson     30435  27041  26656  19730


# Localisation de fontaines

In [44]:
import geopandas as gpd
from shapely.geometry import shape, LineString, mapping
from ipywidgets import Output, VBox, HTML
from ipyleaflet import Map, GeoJSON, GeomanDrawControl, LayersControl, basemaps, basemap_to_tiles

# On travaille UNIQUEMENT avec les coordonnées on_track
REQUIRED_COLS = ["lon_ontrack", "lat_ontrack"]

missing = [c for c in REQUIRED_COLS if c not in bd.columns]
if missing:
    raise ValueError(
        f"Colonnes absentes dans bd : {missing}. "
        f"Je ne peux pas utiliser les coordonnées on_track sans elles."
    )

loc_df = bd.copy()

loc_df = loc_df[
    loc_df["lon_ontrack"].notna() &
    loc_df["lat_ontrack"].notna()
].copy()

if "date" in loc_df.columns:
    loc_df["date"] = pd.to_datetime(loc_df["date"], errors="coerce").dt.date

if "point_id" in loc_df.columns:
    loc_df["point_id"] = pd.to_numeric(loc_df["point_id"], errors="coerce")

if "track_id" in loc_df.columns:
    loc_df["track_id"] = loc_df["track_id"].astype(str).str.lower().str.strip()

gdf_all = gpd.GeoDataFrame(
    loc_df,
    geometry=gpd.points_from_xy(loc_df["lon_ontrack"], loc_df["lat_ontrack"]),
    crs="EPSG:4326"
)

print("Préparation terminée")
print("Nombre de points :", len(gdf_all))
print("Parcours disponibles :", sorted(gdf_all["track_id"].dropna().unique()))

Préparation terminée
Nombre de points : 341281
Parcours disponibles : ['antigone', 'boulevards', 'ecusson']


In [45]:
selected_points_last = pd.DataFrame(columns=["lon_ontrack", "lat_ontrack"])
selected_polygon_last = None
selected_csv_path_last = None

In [46]:
def build_ontrack_zone_selector(
    gdf,
    track="ecusson",
    date_min=None,
    date_max=None,
    m_slots=None,
    display_step=1,
    output_dir=OUT_LOCALISATION,
    output_filename="coords_ontrack_selection.csv"
):
    """
    Dessine une zone sur la carte et exporte UNIQUEMENT un CSV
    contenant les coordonnées on_track :
    - lon_ontrack
    - lat_ontrack

    track peut être :
    - une chaîne : "ecusson"
    - une liste : ["ecusson", "antigone", "boulevards"]
    """

    global selected_points_last, selected_polygon_last, selected_csv_path_last

    data = gdf.copy()

    def normalize_tracks(track_value):
        if track_value is None:
            return None
        if isinstance(track_value, (list, tuple, set, np.ndarray, pd.Series)):
            return [str(t).lower().strip() for t in track_value]
        return [str(track_value).lower().strip()]

    tracks = normalize_tracks(track)

    # Couleur par parcours
    ROUTE_COLORS = {
        "ecusson": "red",
        "antigone": "blue",
        "boulevards": "green"
    }

    # Filtre parcours
    if "track_id" in data.columns and tracks is not None:
        data = data[data["track_id"].isin(tracks)].copy()

    # Filtre dates
    if "date" in data.columns:
        if date_min is not None:
            data = data[data["date"] >= pd.to_datetime(date_min).date()]
        if date_max is not None:
            data = data[data["date"] <= pd.to_datetime(date_max).date()]

    # Filtre créneaux
    if m_slots is not None and "M_slot" in data.columns:
        wanted = {str(x).upper().strip() for x in m_slots}
        data = data[data["M_slot"].astype(str).str.upper().isin(wanted)].copy()

    if data.empty:
        raise ValueError("Aucune donnée après filtrage.")

    sort_cols = [c for c in ["track_id", "date", "M_slot", "point_id"] if c in data.columns]
    if sort_cols:
        data = data.sort_values(sort_cols).copy()

    # Un tracé de référence par parcours
    route_layers = []
    tracks_present = sorted(data["track_id"].dropna().unique()) if "track_id" in data.columns else ["parcours"]

    for trk in tracks_present:
        sub = data[data["track_id"] == trk].copy() if "track_id" in data.columns else data.copy()

        ref_cols = [c for c in ["date", "M_slot"] if c in sub.columns]
        if ref_cols and not sub.empty:
            ref_keys = sub[ref_cols].drop_duplicates().iloc[0].to_dict()
            mask = pd.Series(True, index=sub.index)
            for c, v in ref_keys.items():
                mask &= sub[c] == v
            ref_pass = sub.loc[mask].copy()
        else:
            ref_pass = sub.copy()

        if "point_id" in ref_pass.columns:
            ref_pass = ref_pass.sort_values("point_id")

        ref_pass = ref_pass.iloc[::max(1, display_step)].copy()

        if len(ref_pass) >= 2:
            route_line = LineString(list(zip(ref_pass["lon_ontrack"], ref_pass["lat_ontrack"])))

            route_geojson = {
                "type": "FeatureCollection",
                "features": [{
                    "type": "Feature",
                    "geometry": mapping(route_line),
                    "properties": {"track_id": trk}
                }]
            }

            color = ROUTE_COLORS.get(trk, "orange")

            layer = GeoJSON(
                data=route_geojson,
                style={
                    "color": color,
                    "weight": 3,
                    "opacity": 0.85
                }
            )
            route_layers.append(layer)

    center = [float(data["lat_ontrack"].median()), float(data["lon_ontrack"].median())]

    try:
        base = basemap_to_tiles(basemaps.Esri.WorldImagery)
        m = Map(
            center=center,
            zoom=19,
            max_zoom=24,
            min_zoom=3,
            zoom_snap=0.1,
            zoom_delta=0.25,
            scroll_wheel_zoom=True,
            double_click_zoom=True,
            box_zoom=True,
            touch_zoom=True,
            dragging=True,
            layers=(base,)
        )
    except Exception:
        m = Map(
            center=center,
            zoom=19,
            max_zoom=24,
            min_zoom=3,
            zoom_snap=0.1,
            zoom_delta=0.25,
            scroll_wheel_zoom=True,
            double_click_zoom=True,
            box_zoom=True,
            touch_zoom=True,
            dragging=True
        )

    # Couche de sélection
    selected_layer = GeoJSON(
        data={"type": "FeatureCollection", "features": []},
        style={"color": "cyan", "weight": 2, "opacity": 0.8, "fillOpacity": 0.35}
    )

    for layer in route_layers:
        m.add(layer)

    m.add(selected_layer)
    m.add(LayersControl())

    draw = GeomanDrawControl()
    draw.polygon = {"pathOptions": {"color": "#00FFFF"}}
    draw.rectangle = {"pathOptions": {"color": "#00FFFF"}}
    draw.circle = {}
    draw.polyline = {}
    draw.marker = {}
    draw.circlemarker = {}
    m.add(draw)

    out = Output()

    title_tracks = ", ".join(tracks_present)
    title = HTML(
        f"<b>Parcours :</b> {title_tracks} "
        f"&nbsp;&nbsp; <b>Couleurs :</b> "
        f"<span style='color:red;'>ecusson</span>, "
        f"<span style='color:blue;'>antigone</span>, "
        f"<span style='color:green;'>boulevards</span>"
    )

    def handle_draw(self, action, geo_json):
        global selected_points_last, selected_polygon_last, selected_csv_path_last

        if action not in {"create", "edit", "cut"}:
            return

        feats = geo_json if isinstance(geo_json, list) else [geo_json]
        feat = feats[-1]

        geom = shape(feat["geometry"])
        selected_polygon_last = geom

        # Points inclus dans la zone
        selected = data[data.geometry.intersects(geom)].copy()

        if selected.empty:
            with out:
                out.clear_output()
                print("Aucun point trouvé dans la zone.")
            selected_layer.data = {"type": "FeatureCollection", "features": []}
            selected_points_last = pd.DataFrame(columns=["lon_ontrack", "lat_ontrack"])
            selected_csv_path_last = output_dir / output_filename
            return

        # CSV final : uniquement les coordonnées on_track
        selected_points_last = (
            selected[["lon_ontrack", "lat_ontrack"]]
            .drop_duplicates()
            .sort_values(["lon_ontrack", "lat_ontrack"])
            .reset_index(drop=True)
        )

        # Aperçu sur carte
        preview = selected.iloc[::max(1, display_step)].copy()
        preview_json = preview.copy()
        if "date" in preview_json.columns:
            preview_json["date"] = preview_json["date"].astype(str)

        selected_layer.data = json.loads(preview_json.to_json())

        # Export CSV direct
        csv_path = output_dir / output_filename
        selected_points_last.to_csv(csv_path, index=False)
        selected_csv_path_last = csv_path

        with out:
            out.clear_output()
            print("Sélection terminée.")
            print(f"Nombre de coordonnées on_track uniques : {len(selected_points_last)}")
            print(f"CSV exporté : {csv_path}")
            display(selected_points_last.head(20))

    draw.on_draw(handle_draw)

    return VBox([title, m, out])

In [47]:
ui = build_ontrack_zone_selector(
    gdf_all,
    track=["ecusson", "antigone", "boulevards"],
    date_min="2024-10-01",
    date_max="2025-01-31",
    m_slots=["M1", "M2", "M3", "M4"],
    display_step=1,
    output_dir=OUT_LOCALISATION,
    output_filename="coords_ontrack_fontaine_barnes.csv"
)

ui

In [48]:
print("Aperçu des points sélectionnés :")
display(selected_points_last.head(20))

print("\nNombre total de points sélectionnés :", len(selected_points_last))
print("CSV exporté :", selected_csv_path_last)

Aperçu des points sélectionnés :


,lon_ontrack,lat_ontrack
0,3.877433,43.607992
1,3.877434,43.608006
2,3.877436,43.607991
3,3.877436,43.608015
4,3.877438,43.608023
5,3.877439,43.607998
6,3.877444,43.607989
7,3.877446,43.608029
8,3.877447,43.607985
9,3.877455,43.608036



Nombre total de points sélectionnés : 23
CSV exporté : outputs\localisation\coords_ontrack_fontaine_barnes.csv
